In [8]:
# from google.colab import files
# uploaded = files.upload()

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
import matplotlib.pyplot as plt

In [10]:
# Load data
url = "https://raw.githubusercontent.com/mahmoudramadan155/ML/main/labs/dl/bengaluru_house_prices.csv"

df = pd.read_csv(url)
df.head()

# Handle total_sqft range values
def convert_sqft(x):
    tokens = str(x).split('-')
    if len(tokens) == 2:
        return (float(tokens[0]) + float(tokens[1])) / 2
    try:
        return float(x)
    except:
        return None

df['total_sqft'] = df['total_sqft'].apply(convert_sqft)
df['bhk'] = df['size'].str.extract(r'(\d+)').astype(float)
df['location'] = df['location'].astype(str).str.strip()

In [11]:
# Drop null values
df = df.dropna(subset=['total_sqft', 'bath', 'bhk', 'price', 'location'])

# Filter simple outliers
df = df[~(df.total_sqft / df.bhk < 300)]

# Group less frequent locations
loc_counts = df['location'].value_counts()
other_locs = loc_counts[loc_counts <= 10]
df['location'] = df['location'].apply(lambda x: 'other' if x in other_locs else x)

# One-hot encoding
dummies = pd.get_dummies(df['location'], drop_first=True)
df_final = pd.concat([df[['total_sqft', 'bath', 'bhk', 'price']], dummies], axis=1)

# Features and target
X = df_final.drop('price', axis=1)
y = df_final['price']

In [12]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#  Neural Network Model
model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

In [ ]:
# Training
history = model.fit(
    X_train_scaled, y_train,
    epochs=60,
    batch_size=32,
    validation_split=0.1
)

Epoch 1/60
281/281 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 21869.1035 - mae: 67.1748 - val_loss: 15384.7891 - val_mae: 50.6018
Epoch 2/60
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 14245.9443 - mae: 45.7057 - val_loss: 14160.3682 - val_mae: 46.1744
Epoch 3/60
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 13855.8457 - mae: 43.2138 - val_loss: 13593.9902 - val_mae: 51.5180
Epoch 4/60
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 13474.3926 - mae: 42.8393 - val_loss: 13376.1035 - val_mae: 48.9954
Epoch 5/60
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 13421.2480 - mae: 42.2235 - val_loss: 13238.9199 - val_mae: 45.8640
Epoch 6/60
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 13310.8525 - mae: 41.1723 - val_loss: 13168.9873 - val_mae: 47.2050
Epoch 7/60
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 13112.1514 - mae: 41.3757 - val_loss: 13063.0508 - val_mae: 43.7551
Epoch 8/60
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 13074.2979 - mae: 40.7480 - val_loss: 12709.

In [ ]:
# Plot Training & Validation Loss
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Train Loss (MSE)')
plt.plot(history.history['val_loss'], label='Val Loss (MSE)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Model Loss Over Epochs')
plt.legend()
plt.show()

In [ ]:
# Evaluation
y_pred = model.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R2 Score: {r2 * 100:.2f}%")